#### Imports

In [ ]:
# Make utils/ importable when running this notebook from notebooks/
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
# Third-party
import dtale
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import r2_score

# Shared utils
from utils.checkpoint import Checkpoint
from utils.data_loading import load_run_config
from utils.plotting import plot_scatter
from utils.stats import STATE_CODE_MAP, grouped_error_stats

## 1. Setup

In [ ]:
cp = Checkpoint("Start")

#### Paths and files

In [ ]:
# Same per-run config test.ipynb used
config = load_run_config("D:/.../eval_config.yaml")

year = config["year"]

eval_dir = Path(config["eval_dir"])
input_csv = eval_dir / f"test_predictions_cells_ihse_{year}.csv"

ancillary_census_files = config.get("ancillary_census_files", [])

## 2. Data loading

In [ ]:
input_data = pd.read_csv(input_csv, sep=";", converters={"CellCode": str, "BL": str, "QUALI": str})

In [ ]:
# Ancillary census tables (age classes, area per person, average age, building age, household size, ...)
census_dfs = [pd.read_csv(path, sep=";") for path in ancillary_census_files]

In [ ]:
cp.mark("Data loaded")

## 3. Data preparation

In [ ]:
for df in census_dfs:
    input_data = input_data.merge(df, on="CellCode", how="left")

In [ ]:
input_data["BL_named"] = input_data["BL"].map(STATE_CODE_MAP)

## 4. Plotting

#### General error

In [ ]:
plot_scatter(input_data["Einwohner"], input_data["ERROR"],
            xlabel="Census cell inhabitants", ylabel="Error [inhabitants]",
            title=f"Cell size vs. Error \n (Cell level, limited to 500 inhabitants max.) \n [Test {year}] \n",
            out_path=eval_dir / f"test_cellsize_error_{year}.png",
            xlim=(0, 500), ylim=(-500, 500), diagonal=False)

In [ ]:
plot_scatter(input_data["Einwohner"], input_data["REL_ERROR"],
            xlabel="Census cell inhabitants", ylabel="Rel. Error [%]",
            title=f"Cell size vs. Relative Error \n (Cell level, limited to 500 inhabitants max.) \n [Test {year}] \n",
            out_path=eval_dir / f"test_cellsize_relerror_{year}.png",
            xlim=(0, 500), diagonal=False)

In [ ]:
r2 = r2_score(input_data["Einwohner"], input_data["POPULATION_EST"])
plot_scatter(input_data["Einwohner"], input_data["POPULATION_EST"],
            xlabel="Census inhabitants", ylabel="Predicted inhabitants",
            title=f"Actuals vs. Predictions \n (Cell level, limited to 500 inhabitants max.) \n [Test {year}] \n",
            out_path=eval_dir / f"test_actuals_predictions_{year}.png",
            xlim=(0, 500), ylim=(0, 500), r2=r2, diagonal=True)

In [ ]:
# Actuals vs. predictions for several random samples, to see how consistent R² is across subsets
n_samples = 20
sample_fraction = 0.1
rows, cols = 5, 4
xmax = 500
ymax = 500

fig, axes = plt.subplots(rows, cols, figsize=(16, 20), dpi=120)
axes = axes.flatten()

for i in range(n_samples):
    sample = input_data.sample(frac=sample_fraction)
    r2_sample = r2_score(sample["Einwohner"], sample["POPULATION_EST"])

    ax = axes[i]
    ax.scatter(sample["Einwohner"], sample["POPULATION_EST"], color="darkblue", marker=".", s=10, zorder=2)
    ax.plot([0, xmax], [0, ymax], "k--", linewidth=1, label="x = y")
    ax.set_xlim(0, xmax)
    ax.set_ylim(0, ymax)
    ax.set_title(f"Sample {i+1}", fontsize=10)
    ax.set_xlabel("Census inhabitants", fontsize=10)
    ax.set_ylabel("Predicted inhabitants", fontsize=10)
    ax.tick_params(axis="both", labelsize=8, colors="0.3")
    ax.grid(axis="y", alpha=0.5, zorder=0)
    sns.despine(ax=ax)
    ax.text(0.05, 0.95, f"R² = {r2_sample:.2f}", transform=ax.transAxes,
            fontsize=10, color="red", ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", boxstyle="round,pad=0.2"))

fig.suptitle(f"Actuals vs. Predictions for {n_samples} random {int(sample_fraction*100)}% samples \n (Cell level, limited to 500 inhabitants max.) \n [Test {year}] \n", fontsize=12, color="0")
plt.tight_layout()
plt.savefig(eval_dir / f"test_actuals_predictions_random_{year}.png", dpi=300, bbox_inches="tight")
plt.show()

#### Vs. census data

In [ ]:
plot_scatter(input_data["durchschnFlaechejeBew"], input_data["ERROR"],
            xlabel="Mean living area per inhabitant [m²]", ylabel="Error [inhabitants]",
            title=f"Mean living area vs. Error \n (Cell level) \n [Test {year}] \n",
            out_path=eval_dir / f"test_area_per_person_error_{year}.png",
            xlim=(0, 500), ylim=(-500, 500), diagonal=False)

In [ ]:
plot_scatter(input_data["Durchschnittsalter"], input_data["ERROR"],
            xlabel="Mean inhabitants age [years]", ylabel="Error [inhabitants]",
            title=f"Mean inhabitants age vs. Error \n (Cell level) \n [Test {year}] \n",
            out_path=eval_dir / f"test_average_age_error_{year}.png",
            xlim=(0, 120), ylim=(-500, 500), diagonal=False)

In [ ]:
plot_scatter(input_data["DurchschnHHGroesse"], input_data["ERROR"],
            xlabel="Mean household size [inhabitants]", ylabel="Error [inhabitants]",
            title=f"Mean household size vs. Error \n (Cell level) \n [Test {year}] \n",
            out_path=eval_dir / f"test_household_size_error_{year}.png",
            xlim=(0, 50), ylim=(-500, 500), diagonal=False)

#### Federal states

In [ ]:
# Actuals vs. predictions scatterplot per federal state
xmax = 500
ymax = 500

fig, axes = plt.subplots(4, 4, figsize=(16, 16), dpi=120)
axes = axes.flatten()

bl_categories = sorted(input_data["BL_named"].dropna().unique())

for i, bl in enumerate(bl_categories[:16]):
    ax = axes[i]
    subset = input_data[input_data["BL_named"] == bl]

    if len(subset) > 1:
        r2_bl = r2_score(subset["Einwohner"], subset["POPULATION_EST"])
        r2_text = f"R² = {r2_bl:.2f}"
    else:
        r2_text = "R² = NA"

    ax.scatter(subset["Einwohner"], subset["POPULATION_EST"], color="darkblue", marker=".", s=10, zorder=2)
    ax.plot([0, xmax], [0, ymax], "k--", label="x = y")
    ax.set_xlim(0, xmax)
    ax.set_ylim(0, ymax)
    ax.set_title(f"{bl}", fontsize=11, color="0.2")
    ax.set_xlabel("Census inhabitants", fontsize=10)
    ax.set_ylabel("Predicted inhabitants", fontsize=10)
    ax.tick_params(axis="both", labelsize=8, colors="0.3")
    ax.grid(axis="y", alpha=0.75, zorder=0)
    sns.despine(ax=ax)
    ax.text(0.05, 0.95, r2_text, transform=ax.transAxes,
            fontsize=10, color="red", ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", boxstyle="round,pad=0.2"))

for j in range(len(bl_categories), 16):
    fig.delaxes(axes[j])

fig.suptitle(f"Actuals vs. Predictions per federal state \n (Cell level, limited to 500 inhabitants max.) \n [Test {year}] \n", fontsize=12, color="0")
plt.tight_layout()
plt.savefig(eval_dir / f"test_actuals_predictions_BL_grid_{year}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Quality classes per state
percent_df = pd.crosstab(input_data["BL_named"], input_data["QUALI"], normalize="index") * 100
percent_df = percent_df.sort_index(ascending=False)
percent_df = percent_df[sorted(percent_df.columns)]

fig, ax = plt.subplots(figsize=(10, 8), dpi=120)
percent_df.plot(kind="barh", stacked=True, colormap="RdYlGn_r", width=0.7, ax=ax, legend=False)
plt.title(f"Quality class distribution per federal state \n (Cell level) \n [Test {year}] \n", fontsize=12, color="0")
ax.set_xlabel("Percentage [%]", fontsize=12, color="0")
ax.set_ylabel("Federal state", fontsize=12, color="0")
plt.xticks(fontsize=10, color="0.3")
plt.yticks(fontsize=10, color="0.3")

for i, row in enumerate(percent_df.values):
    left = 0
    for val in row:
        if val > 0:
            ax.text(left + val / 2, i, f"{val:.0f}%", ha="center", va="center", fontsize=9, color="black")
        left += val

ax.legend(title="Quality class", bbox_to_anchor=(0.97, 1), loc="upper left")
sns.despine()
plt.savefig(eval_dir / f"test_quality_states_{year}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
cp.mark("Plotting done")

## 5. Tables

In [ ]:
age_classes_grouped = grouped_error_stats(input_data, "Altersklasse")
age_classes_grouped

In [ ]:
building_age_grouped = grouped_error_stats(input_data, "BaujahrJZ")
building_age_grouped

In [ ]:
age_classes_grouped.to_csv(eval_dir / f"test_statistics_age_classes_{year}.csv", sep=";", decimal=",", index=False)
building_age_grouped.to_csv(eval_dir / f"test_statistics_building_age_{year}.csv", sep=";", decimal=",", index=False)

#### Script runtime

In [ ]:
cp.finish()